# 冻结SOXL/SOXX数据基准审计

主证据来自QLib。首次读取只允许指定冻结worktree且manifest必须与b94059c2e0754b0ab102e4f62534eefb一致；随后只读专属审计Recorder。主目录实时行情、旧模型和旧provider均不修改。

此notebook核验复权基准和OHLC物理一致性，不进行新模型拟合或组合搜索。

In [1]:
from pathlib import Path
import os,sys,json,hashlib
sys.dont_write_bytecode=True
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
FROZEN_ROOT=Path('/Users/rziiiii/dev/workspace-worktrees/convex-excess-20261002')
CANONICAL_ROOT=Path('/Users/rziiiii/dev/workspace')
REPO_ROOT=Path(os.environ.get('SOXL_DATA_AUDIT_REPOSITORY',str(FROZEN_ROOT if FROZEN_ROOT.exists() else CANONICAL_ROOT))).resolve()
CARD=REPO_ROOT/'research/convex_excess_20261002'
SUMMARY_PATH=CARD/'data_quality_summary.json'
NOTEBOOK=Path('/tmp/soxl_data_basis_audit.ipynb')
SOURCE_ID='b94059c2e0754b0ab102e4f62534eefb'
EXPERIMENT='convex_excess_20261002'
RECORDER_URI=CANONICAL_ROOT/'research'/EXPERIMENT/'qlib_recorder'
sys.path.insert(0,str(REPO_ROOT))
import numpy as np,pandas as pd,qlib,nbformat
from qlib.workflow import R
from src.data import QlibProviderConfig,load_qlib_feature_frame
from src.data.integrated_allocation import frame_sha256

def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def ready(value):
    if isinstance(value,dict):return {str(k):ready(v) for k,v in value.items()}
    if isinstance(value,(tuple,list,np.ndarray)):return [ready(v) for v in value]
    if isinstance(value,(np.integer,np.bool_)):return value.item()
    if isinstance(value,(np.floating,float)):return float(value) if np.isfinite(value) else None
    if isinstance(value,pd.Timestamp):return value.isoformat()
    return value

def atomic_json(path,value):
    temporary=path.with_suffix('.tmp');temporary.write_text(json.dumps(ready(value),ensure_ascii=False,indent=2,allow_nan=False)+'\n');temporary.replace(path)

code_sha=hashlib.sha256('\n'.join(c.source for c in nbformat.read(NOTEBOOK,as_version=4).cells if c.cell_type=='code').encode()).hexdigest()
source_summary=json.loads((CANONICAL_ROOT/'research/stable_excess_integration_20261002/summary.json').read_text())
assert source_summary['recorder_id']==SOURCE_ID
# This is the named immutable execution snapshot, never the live operational
# soxl/soxx providers. It only bootstraps the native Qlib Recorder interface.
bootstrap=REPO_ROOT/source_summary['provider_uri']
boot_manifest=json.loads((bootstrap/'qlib_store_manifest.json').read_text())
assert boot_manifest['as_of']=='2026-09-25'
assert boot_manifest['source']=='Qlib frozen execution frame '+source_summary['data_audit']['execution_prices_sha256']
qlib.init(provider_uri=str(bootstrap),region='us',kernels=1)
previous=json.loads(SUMMARY_PATH.read_text()) if SUMMARY_PATH.exists() else None
if previous:
    assert previous['source_recorder_id']==SOURCE_ID
    extension=previous['analysis_code_sha256']!=code_sha
    if extension and os.environ.get('SOXL_DATA_AUDIT_EXTEND')!='1':raise RuntimeError('Changed analysis requires explicit new audit revision')
    with R.uri_context(previous['audit_recorder_uri']):
        audit_recorder=R.get_recorder(experiment_name=EXPERIMENT,recorder_id=previous['audit_recorder_id'])
        recdir=Path(audit_recorder.get_local_dir())
        for artifact,digest in previous['immutable_artifact_sha256'].items():
            assert sha(recdir/'artifacts'/artifact)==digest,(artifact,'immutable artifact changed')
        frozen=audit_recorder.load_object('data_quality/frozen_inputs.pkl')
        stored_summary=audit_recorder.load_object('data_quality/numeric_summary.pkl')
    source_mode='immutable_audit_recorder'
else:
    if os.environ.get('SOXL_DATA_AUDIT_FORCE_CACHE')=='1':raise RuntimeError('blocked: requested immutable audit cache is absent')
    if REPO_ROOT!=FROZEN_ROOT.resolve() or not FROZEN_ROOT.exists():
        raise RuntimeError('blocked: initial audit requires the explicit frozen worktree; never read live main providers')
    with R.uri_context(source_summary['recorder_uri']):
        original=R.get_recorder(experiment_name='stable_excess_integration_20261002',recorder_id=SOURCE_ID)
        source_audit=original.load_object('inputs/source_audit.pkl')
        original_features=original.load_object('inputs/primary_frame.pkl')
        original_prices=original.load_object('inputs/execution_prices.pkl')
    assert frame_sha256(original_features)==source_audit['primary_frame_sha256']
    assert frame_sha256(original_prices)==source_audit['execution_prices_sha256']
    frames={};manifests={};code={}
    for role,name,fields in [('primary','soxl',['$open','$high','$low','$close','$volume']),('execution','soxl_soxx_soxs',['$open','$high','$low','$close'])]:
        provider=FROZEN_ROOT/'data/qlib'/name;path=provider/'qlib_store_manifest.json'
        expected=source_audit[role]['manifest_sha256'];assert sha(path)==expected
        manifest=json.loads(path.read_text());assert manifest['as_of']=='2026-09-25'
        frame=load_qlib_feature_frame(QlibProviderConfig(provider_uri=provider),instruments=['SOXL','SOXX'],fields=fields,start='2010-03-11',end='2026-09-25')
        assert sha(path)==expected,'Provider changed during native read'
        checked=frame if role=='primary' else frame.loc[:,['$open','$close']]
        assert frame_sha256(checked)==source_audit[role]['readback_sha256']
        frames[role]=frame
        manifests[role]={'manifest':manifest,'manifest_sha256':expected,'native_frame_sha256':frame_sha256(frame)}
    for name in ['src/data/market_update.py','src/data/soxs_defensive.py','src/data/integrated_allocation.py']:
        p=FROZEN_ROOT/name;code[name]={'sha256':sha(p),'text':p.read_text()}
    futu_path=FROZEN_ROOT/'data/raw/external/futu_history_kline_US_SOXL.csv'
    futu=pd.read_csv(futu_path)
    futu_row=futu.loc[futu.date.eq('2023-06-05'),['date','symbol','open','high','low','close','volume']].copy()
    assert len(futu_row)==1
    frozen={'source_recorder_id':SOURCE_ID,'source_as_of':'2026-09-25','source_root':str(FROZEN_ROOT),
        'primary_ohlcv':frames['primary'],'execution_ohlc':frames['execution'],
        'original_feature_frame':original_features,'original_execution_prices':original_prices,
        'source_audit':source_audit,'provider_metadata':manifests,
        'execution_raw_source_manifest':json.loads((FROZEN_ROOT/'data/raw/soxs_defensive_v1/source_manifest.json').read_text()),
        'source_code':code,'retained_futu_corroboration':futu_row,'retained_futu_file_sha256':sha(futu_path),
        'retained_futu_file_last_date':str(futu.date.max()),
        'unarchived_external_review':'A sibling agent reports a fresh Yahoo chart row matching retained primary June5 OHLC; no raw response, query time or payload hash was retained, so this is not replayable source evidence and is not used in numeric conclusions.'}
    source_mode='frozen_worktree_native_qlib'
    extension=False

primary=frozen['primary_ohlcv'];execution=frozen['execution_ohlc'];features=frozen['original_feature_frame'];prices=frozen['original_execution_prices']
assert primary.index.equals(execution.index) and not primary.index.has_duplicates
assert np.isfinite(primary.to_numpy()).all() and np.isfinite(execution.to_numpy()).all()
assert (primary['$volume']>0).all()
assert frame_sha256(primary)==frozen['provider_metadata']['primary']['native_frame_sha256']
assert frame_sha256(execution)==frozen['provider_metadata']['execution']['native_frame_sha256']
errors={}
for symbol in ['SOXL','SOXX']:
    for field in ['open','close']:
        got=execution.xs(symbol,level='instrument')['$'+field]
        assert got.index.equals(prices.index)
        errors[symbol+'_'+field]=float(np.max(np.abs(got.to_numpy()-prices[(field,symbol)].to_numpy())))
assert max(errors.values())==0
parts=[]
for symbol in ['SOXL','SOXX']:
    p=primary.xs(symbol,level='instrument');e=execution.xs(symbol,level='instrument')
    part=pd.DataFrame({'instrument':symbol,'primary_close':p['$close'],'execution_close':e['$close'],
        'primary_close_return':p['$close'].pct_change(fill_method=None),'execution_close_return':e['$close'].pct_change(fill_method=None)})
    part['return_difference']=part.primary_close_return-part.execution_close_return
    part['close_ratio']=part.primary_close/part.execution_close
    parts.append(part.reset_index())
basis=pd.concat(parts,ignore_index=True)
soxx=basis[basis.instrument.eq('SOXX')].set_index('datetime').loc[features.index]
feature_ret_error=float(np.max(np.abs(soxx.primary_close_return.to_numpy()-features.soxx_ret1.to_numpy())))
assert feature_ret_error==0
annual=soxx.groupby(soxx.index.year).agg(max_abs_return_difference=('return_difference',lambda x:x.abs().max()),
    mean_abs_return_difference=('return_difference',lambda x:x.abs().mean()),ratio_first=('close_ratio','first'),ratio_last=('close_ratio','last')).reset_index().rename(columns={'datetime':'year'})
# Price fields are stored float32; perform the physical-bar comparison in
# float64 so a tiny positive difference is not rounded away by a ratio.
e=execution.astype('float64').copy()
e['low_violation']=np.maximum(e['$low']/e[['$open','$close']].min(axis=1)-1,0)
e['high_violation']=np.maximum(e[['$open','$close']].max(axis=1)/e['$high']-1,0)
e['severity']=e[['low_violation','high_violation']].max(axis=1)
violations=e[e.severity.gt(0)].copy()
# Data-only uniform adjustment candidate; never an implicit execution repair.
p_original=primary.loc[:,['$open','$high','$low','$close']].astype('float64')
p_checked=p_original.copy()
p_checked['low_violation']=np.maximum(p_checked['$low']/p_checked[['$open','$close']].min(axis=1)-1,0)
p_checked['high_violation']=np.maximum(p_checked[['$open','$close']].max(axis=1)/p_checked['$high']-1,0)
p_checked['severity']=p_checked[['low_violation','high_violation']].max(axis=1)
primary_violations=p_checked[p_checked.severity.gt(0)].copy()
factor=e['$close']/p_original['$close'];assert np.isfinite(factor).all() and factor.gt(0).all()
uniform_candidate=p_original.mul(factor,axis=0)
uniform_candidate['$volume']=primary['$volume']
np.testing.assert_array_equal(uniform_candidate['$volume'].to_numpy(),primary['$volume'].to_numpy())
open_differences=pd.DataFrame({'factor':factor,'adjusted_primary_open':uniform_candidate['$open'],
    'frozen_execution_open':e['$open'],'price_difference':uniform_candidate['$open']-e['$open'],
    'relative_difference':uniform_candidate['$open']/e['$open']-1})
open_differences['absolute_relative_difference']=open_differences.relative_difference.abs()
open_by_instrument=[]
for symbol,g in open_differences.groupby(level='instrument'):
    worst=g.absolute_relative_difference.idxmax()
    open_by_instrument.append({'instrument':symbol,'rows':len(g),'maximum_absolute_price_difference':float(g.price_difference.abs().max()),
        'maximum_absolute_relative_difference':float(g.absolute_relative_difference.max()),'worst_date':str(worst[1].date()),
        'above_relative_threshold_counts':{str(t):int(g.absolute_relative_difference.gt(t).sum()) for t in [1e-7,1e-6,1e-5,1e-4,.001]}})
material=e[e.severity.gt(1e-5)].copy()
assert len(material)==1 and material.index[0]==('SOXL',pd.Timestamp('2023-06-05'))
material_row=material.reset_index().iloc[0].to_dict()
example=soxx.loc[pd.Timestamp('2014-12-24')].to_dict()
example['date']='2014-12-24'
prior_day=soxx.index[soxx.index.get_loc(pd.Timestamp('2014-12-24'))-1]
example['prior_primary_close']=float(soxx.loc[prior_day,'primary_close'])
example['prior_execution_close']=float(soxx.loc[prior_day,'execution_close'])
observations={'instrument_session_rows':len(primary),'calendar_start':str(prices.index.min().date()),'calendar_end':str(prices.index.max().date()),
    'primary_fields':list(primary.columns),'execution_fields':list(execution.columns),'execution_has_volume':'$volume' in execution,
    'nonpositive_volume_rows':int(primary['$volume'].le(0).sum()),'native_open_close_max_errors_vs_source_recorder':errors,
    'feature_rows':len(soxx),'source_feature_ret_reconstruction_max_error':feature_ret_error,
    'soxx_close_return_max_absolute_difference':float(soxx.return_difference.abs().max()),
    'soxx_close_return_difference_above_1e_minus7_rows':int(soxx.return_difference.abs().gt(1e-7).sum()),
    'soxx_example':example,'low_above_open_or_close_rows':int((e['$low']>e[['$open','$close']].min(axis=1)).sum()),
    'high_below_open_or_close_rows':int((e['$high']<e[['$open','$close']].max(axis=1)).sum()),
    'physical_violation_precision':'float64 comparisons of native float32 stored observations',
    'relative_violation_counts':{str(t):int(e.severity.gt(t).sum()) for t in [0.,1e-7,1e-6,1e-5,1e-4,.001]},
    'material_ohlc_bar':material_row,'primary_same_day_bar':primary.loc[('SOXL',pd.Timestamp('2023-06-05'))].to_dict(),
    'primary_frame_sha256':frame_sha256(primary),'execution_frame_sha256':frame_sha256(execution),
    'basis_analysis_sha256':frame_sha256(basis),'violation_analysis_sha256':frame_sha256(violations)}
observations['primary_ohlc_audit']={'low_above_open_or_close_rows':int((p_checked['$low']>p_checked[['$open','$close']].min(axis=1)).sum()),
    'high_below_open_or_close_rows':int((p_checked['$high']<p_checked[['$open','$close']].max(axis=1)).sum()),
    'maximum_relative_violation':float(p_checked.severity.max()),
    'relative_violation_counts':{str(t):int(p_checked.severity.gt(t).sum()) for t in [0.,1e-7,1e-6,1e-5,1e-4,.001]}}
observations['uniform_adjustment_candidate']={'status':'data-only candidate; not accepted or applied to execution',
    'formula':'one positive execution_close/primary_close factor applied identically to PRIMARY O/H/L/C; primary volume unchanged',
    'factor_min':float(factor.min()),'factor_max':float(factor.max()),
    'maximum_close_difference':float(np.abs(uniform_candidate['$close']-e['$close']).max()),
    'volume_unchanged':True,'open_comparison':open_by_instrument,
    'candidate_frame_sha256':frame_sha256(uniform_candidate),'open_comparison_frame_sha256':frame_sha256(open_differences)}
observations=ready(observations)
if previous:
    for key,value in stored_summary.items():assert observations[key]==value,('Immutable source reanalysis differs',key)
    for key,value in previous['observations'].items():assert observations[key]==value,('Prior summary differs',key)
if previous and not extension:
    audit_id=previous['audit_recorder_id'];immutable_hashes=previous['immutable_artifact_sha256']
else:
    RECORDER_URI.mkdir(parents=True,exist_ok=True)
    with R.start(experiment_name=EXPERIMENT,recorder_name='frozen_data_basis_audit',uri=str(RECORDER_URI)):
        audit_recorder=R.get_recorder()
        audit_recorder.log_params(evidence_role='data_quality_only',source_recorder_id=SOURCE_ID,source_as_of='2026-09-25',
            market_fit_performed=False,portfolio_replay_performed=False,analysis_code_sha256=code_sha)
        audit_recorder.save_objects(artifact_path='data_quality',**{'frozen_inputs.pkl':frozen,
            'price_basis_differences.pkl':basis,'annual_basis_summary.pkl':annual,
            'ohlc_violations.pkl':violations,'numeric_summary.pkl':observations,
            'primary_ohlc_violations.pkl':primary_violations,'uniform_adjustment_candidate_ohlcv.pkl':uniform_candidate,
            'uniform_adjustment_open_differences.pkl':open_differences})
        audit_id=audit_recorder.id;recdir=Path(audit_recorder.get_local_dir())
    artifacts=['frozen_inputs.pkl','price_basis_differences.pkl','annual_basis_summary.pkl','ohlc_violations.pkl','numeric_summary.pkl',
        'primary_ohlc_violations.pkl','uniform_adjustment_candidate_ohlcv.pkl','uniform_adjustment_open_differences.pkl']
    immutable_hashes={'data_quality/'+name:sha(recdir/'artifacts/data_quality'/name) for name in artifacts}

summary={'schema_version':2,'parent_audit_recorder_id':previous['audit_recorder_id'] if previous and extension else (previous.get('parent_audit_recorder_id') if previous else None),'evidence_role':'data_quality_only_not_a_model_or_promotion',
    'status':'needs_data_qualification_before_new_basis_rebuild','source_recorder_id':SOURCE_ID,'source_as_of':'2026-09-25',
    'audit_experiment':EXPERIMENT,'audit_recorder_id':audit_id,'audit_recorder_uri':str(RECORDER_URI),
    'audit_recorder_local_dir':str(recdir),'analysis_code_sha256':code_sha,
    'immutable_artifact_sha256':immutable_hashes,'observations':observations,
    'readback_contract':'Subsequent execution loads immutable audit_recorder_id artifacts and verifies byte/content hashes; it never rereads mutable main primary/execution providers.',
    'scope':{'old_stores_changed':False,'models_fitted':False,'portfolio_replayed':False,
      'old_portfolio_uses_execution_open_close':True,'old_features_use_separate_primary_ohlcv':True,
      'phase3_prior_h5_uses_execution_labels_and_opens_not_primary_features':True,
      'old_portfolio_pnl_impact_of_feature_basis_rebuild':'not measured; do not infer causation',
      'interpretation':'Internal recorder/native agreement is not a certificate of coherent adjustment or correct source prices.'},
    'official_index_sources':[
      {'event':'SOXX predecessor changed S&P North American Technology-Semiconductors to PHLX effective2010-10-15', 'url':'https://www.sec.gov/Archives/edgar/data/1100663/000119312510241547/d497.htm'},
      {'event':'SOXX changed PHLX to ICE effective2021-06-21; ICE renamed NYSE2023-11-03','url':'https://www.ishares.com/us/literature/prospectus/p-ishares-phlx-semiconductor-etf-3-31.pdf'},
      {'event':'SOXL changed PHLX to ICE at2021-08-25 market open','url':'https://www.nyse.com/publicdocs/nyse/markets/nyse-arca/rule-interpretations/2021/NYSE%20Arca%20Equities%20RB-21-075.pdf'}],
    'notebook_registration':previous.get('notebook_registration',{'status':'pending_executed_notebook_upload'}) if previous and not extension else {'status':'pending_executed_notebook_upload'}}
CARD.mkdir(parents=True,exist_ok=True)
atomic_json(SUMMARY_PATH,summary)
o=observations;m=o['material_ohlc_bar'];bar=o['primary_same_day_bar'];ex=o['soxx_example']
text=f'''# 冻结数据基准审计

本审计只核验数据，不训练模型、不运行新组合、不修复旧provider。状态：`{summary['status']}`。

来源 Qlib recorder：`{SOURCE_ID}`，as_of2026-09-25。专属审计 recorder：`{audit_id}`。后续重执行从该审计recorder读取冻结输入并核对哈希，不把主目录中持续更新的行情当成冻结数据。

## 已核验事实

- 通过Qlib原生接口读取{o['instrument_session_rows']}条标的交易日记录，区间{o['calendar_start']}至{o['calendar_end']}；价格源和成交量源日期逐项一致。
- 执行provider字段为OHLC，没有volume；primary的volume完整且正值。原生执行O/C与原recorder的冻结执行价格最大误差为0。
- 保存的soxx_ret1与primary收盘收益重建误差为{o['source_feature_ret_reconstruction_max_error']}。
- {o['feature_rows']}条主特征记录中，{o['soxx_close_return_difference_above_1e_minus7_rows']}条收盘收益与执行源差异超过1e-7；最大差异{o['soxx_close_return_max_absolute_difference']*10000:.4f}bp。
- 2014-12-24，primary收盘从{ex['prior_primary_close']:.7f}到{ex['primary_close']:.7f}，收益{ex['primary_close_return']*100:.4f}%；执行源从{ex['prior_execution_close']:.7f}到{ex['execution_close']:.7f}，收益{ex['execution_close_return']*100:.4f}%。
- OHLC物理检查发现{o['low_above_open_or_close_rows']}条low高于open/close、{o['high_below_open_or_close_rows']}条high低于open/close。按原生float32数值转float64计算相对幅度：>1e-6为{o['relative_violation_counts']['1e-06']}条，>1e-5为{o['relative_violation_counts']['1e-05']}条。

唯一超过1e-5的物理异常为{m['instrument']}，{str(m['datetime'])[:10]}：执行源O={m['$open']:.7f}、H={m['$high']:.7f}、L={m['$low']:.7f}、C={m['$close']:.7f}，开盘高于最高价{m['high_violation']*100:.4f}%。同日primary原报价O={bar['$open']:.7f}、H={bar['$high']:.7f}、L={bar['$low']:.7f}、C={bar['$close']:.7f}。冻结Futu旁证及其原文件哈希也已归档；未归档原响应的外部复查只标记为审阅说明，不当作可重放机器证据。

## 来源解释与未解决事项

原代码的Yahoo quote OHLC路径没有应用adjclose/close分红调整；执行provider则显式应用该比例。当前top-level manifest不能证明primary全历史每行都来自同一调整方式。两条SOXX价格流的基准不同，不是同一个SOXX基金在同日跟踪两个指数。

旧Yahoo/Nasdaq拼接按OHLC字段分别估计缩放比例，可能造成细小蜡烛边界破坏。单个2023-06-05 high错误明显更大，初始原始payload尚未完整找回，不能靠high=max(open,close)等补丁捏造价格。旧source代码文本、provider manifest及原执行源manifest随冻结输入归档。

## 同一因子调整PRIMARY的可行性检查

PRIMARY自身物理检查：low高于open/close为{o['primary_ohlc_audit']['low_above_open_or_close_rows']}条，high低于open/close为{o['primary_ohlc_audit']['high_below_open_or_close_rows']}条，最大相对违反{o['primary_ohlc_audit']['maximum_relative_violation']:.10g}。

只在内存中构造候选：对PRIMARY的O/H/L/C使用同一正因子execution_close/primary_close，volume完全不变；没有修改执行O/C或重新计算旧P/L。候选收盘与执行收盘最大浮点差为{o['uniform_adjustment_candidate']['maximum_close_difference']:.10g}。

调整后开盘与冻结执行开盘逐项比较：{json.dumps(o['uniform_adjustment_candidate']['open_comparison'],ensure_ascii=False)}。因此相同收盘基准不自动证明开盘也相同；是否仍有两个来源通道、差异是否可接受及来源需要单独审定。候选整表和开盘差异都归档为数据审计产物，不代表已经批准新的canonical数据。

## 对历史研究的影响边界

原阶段组合结算读取执行O/C，没有直接消费这条错误high；该日期也属于2023外层诊断。原特征使用另外的primary OHLC。Phase3 prior_h5只依据执行收益标签和开盘协方差，不能把其七个开发年份落后全部归咎于特征复权差异。

但特征和标签的价格基准未统一，学习模型及PIT辅助信息的经济解释需在明确重建后复查。影响方向和P/L变化本次没有测量。原数据上的数值可复现，不等于外部行情已验证、数据基准已经合格或存在稳定超额。

## 官方指数历史：另一个应分开的因素

SOXX前身于2010-10-15从S&P North American Technology-Semiconductors改为PHLX，[发行人文件](https://www.sec.gov/Archives/edgar/data/1100663/000119312510241547/d497.htm)有记录。SOXX于2021-06-21改为ICE；ICE于2023-11-03更名NYSE，[iShares招募书](https://www.ishares.com/us/literature/prospectus/p-ishares-phlx-semiconductor-etf-3-31.pdf)说明历史切换。SOXL则于2021-08-25开盘生效，[NYSE监管公告](https://www.nyse.com/publicdocs/nyse/markets/nyse-arca/rule-interpretations/2021/NYSE%20Arca%20Equities%20RB-21-075.pdf)确认。

因此2021-06-21至8/24存在指数错位，名义3倍不是ETF收益恒等式；这也不能单独解释全部年份。现有ETF实际收益上没有再扣一笔虚构的融资成本。

## 下一步

先按来源和物理一致性确定新、独立版本的统一OHLCV基准，必要的分红调整必须对同一bar四个价格用相同因子，并明确volume来源/拆股口径。若无法恢复原高低价，不将执行源OHLC自动当作干净替代品。重建选择不得依据哪个旧/新回测更漂亮。

数据验收后，再以相同Ridge10、相同特征定义和成熟时序重建OOF头；旧模型、provider和证据保持原身份。现金收益仍需独立的真实可得来源合同，不能临时填入收益率使候选过门。稳定超额目标尚未达成。

完整原生frames和明细位于本审计recorder的`data_quality/`；卡根只保留本说明和data_quality_summary.json。
'''
(CARD/'data_quality.md').write_text(text)
print(json.dumps({'audit_recorder_id':audit_id,'source_mode':source_mode,'native_source_providers_read':not bool(previous),
    'feature_rows':o['feature_rows'],'maximum_basis_difference_bp':o['soxx_close_return_max_absolute_difference']*10000,
    'relative_bar_violation_counts':o['relative_violation_counts'],'summary_path':str(SUMMARY_PATH)},ensure_ascii=False,indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[72866:MainThread](2026-10-02 20:05:48,925) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[72866:MainThread](2026-10-02 20:05:48,926) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[72866:MainThread](2026-10-02 20:05:48,926) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/convex-excess-20261002/data/qlib/stable_excess_integration_20261002')}


{
  "audit_recorder_id": "f77655b5e36e4771a7ead1343c287e37",
  "source_mode": "immutable_audit_recorder",
  "native_source_providers_read": false,
  "feature_rows": 3956,
  "maximum_basis_difference_bp": 80.45315742492676,
  "relative_bar_violation_counts": {
    "0.0": 360,
    "1e-07": 224,
    "1e-06": 190,
    "1e-05": 1,
    "0.0001": 1,
    "0.001": 1
  },
  "summary_path": "/Users/rziiiii/dev/workspace-worktrees/convex-excess-20261002/research/convex_excess_20261002/data_quality_summary.json"
}


## 中文结论

已核验并归档8324条冻结原生OHLCV/OHLC来源记录，审计recorder为 `f77655b5e36e4771a7ead1343c287e37`。随后真实重执行已通过：直接读取该recorder并核验源产物哈希，未重新读取主目录实时行情。

发现特征与执行价格复权基准不一致，以及执行SOXL2023-06-05最高价低于开盘价。PRIMARY自身OHLC物理检查通过；同一因子调整PRIMARY全OHLC后，开盘仍与冻结执行开盘存在7条超过0.1%的差异，说明不能仅靠匹配收盘就认定数据已统一。

本次没有修改旧provider、执行O/C、模型或任何账户，没有运行新组合。原结算不使用这条错误high；Phase3 prior_h5不使用primary预测特征，因此不能把原均值控制七年落后归因于特征基准问题。数据修复的模型与P/L影响尚未测量。

下一步先按可核查来源审定新的价格基准，建立独立数据版本；禁止比较旧/新P/L来挑数据。稳定超额目标未达成，旧回放可复现也不构成外部行情正确或科学晋升证明。完整范围与限制见卡根 `data_quality.md` 和 `data_quality_summary.json`。
